In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from itables import show
import ipywidgets as widgets
import json
from IPython.display import HTML, display
import altair as alt
import numpy as np
import uuid
import textwrap
import altair as alt

import warnings
warnings.filterwarnings("ignore")

data_dir = Path('./data')

df_composition = pd.read_excel(data_dir / 'household_composition_ward.xlsx')
df_size = pd.read_excel(data_dir / 'household_size_ward.xlsx')

<h1>HDRC055 - Household Composition</h1>

**Author:** Yanhua Xu

# Category coverage

::: {.callout-note title="Show category coverage details" collapse="true"}

## Household composition

In [2]:
cat_cols = ["main_category", "sub_category1", "sub_category2"]
census_col = "census"

category_presence = (
    df_composition[cat_cols + [census_col]]
    .drop_duplicates()
    .groupby(cat_cols, dropna=False)[census_col]
    .agg(
        presence=lambda x: sorted(x.astype(str).unique()),
        # n_census=lambda x: x.nunique()
    )
    .reset_index()
)

# display(category_presence)

In [3]:
first_category = "Total: All households"
cat_presence_plot = (
    df_composition[["class_name", "main_category", "sub_category1", "sub_category2", "census"]]
    .drop_duplicates()
    .copy()
)

cat_presence_plot["census"] = cat_presence_plot["census"].astype(str)

# clean NA for sorting
cat_presence_plot["sub_category1"] = cat_presence_plot["sub_category1"].fillna("")
cat_presence_plot["sub_category2"] = cat_presence_plot["sub_category2"].fillna("")

# create wrapped label for long category names
cat_presence_plot["class_name_wrapped"] = cat_presence_plot["class_name"].apply(
    lambda x: "\n".join(textwrap.wrap(str(x), width=45))
)

# fixed y-axis order
y_order = (
    cat_presence_plot
    .sort_values(["main_category", "sub_category1", "sub_category2", "class_name"], ascending=True)
    ["class_name_wrapped"]
    .drop_duplicates()
    .tolist()
)

y_order = [first_category] + [x for x in y_order if x not in first_category]

chart = (
    alt.Chart(cat_presence_plot)
    .mark_circle(size=130)
    .encode(
        x=alt.X(
            "census:O",
            title="Census",
            sort=["2001", "2011", "2021"],
            axis=alt.Axis(labelAngle=0)
        ),
        y=alt.Y(
            "class_name_wrapped:N",
            title=None,
            sort=y_order,
            axis=alt.Axis(
                labelLimit=500,
                labelFontSize=10
            )
        ),
        color=alt.Color(
            "main_category:N",
            title="Main category"
        ),
        tooltip=[
            alt.Tooltip("class_name:N", title="Category"),
            alt.Tooltip("main_category:N", title="Main category"),
            alt.Tooltip("sub_category1:N", title="Sub-category 1"),
            alt.Tooltip("sub_category2:N", title="Sub-category 2"),
            alt.Tooltip("census:O", title="Census")
        ]
    )
    .properties(
        width=400,
        height=700,
        title="Category availability by Census"
    )
)

chart

alt.Chart(...)

## Household size

In [4]:
cat_presence_plot = (
    df_size[["class_name", "census"]]
    .drop_duplicates()
    .copy()
)

cat_presence_plot["census"] = cat_presence_plot["census"].astype(str)

# create wrapped label for long category names
cat_presence_plot["class_name_wrapped"] = cat_presence_plot["class_name"].apply(
    lambda x: "\n".join(textwrap.wrap(str(x), width=45))
)

# fixed y-axis order
y_order = (
    cat_presence_plot
    .sort_values(["class_name"], ascending=True)
    ["class_name_wrapped"]
    .drop_duplicates()
    .tolist()
)

y_order = [first_category] + [x for x in y_order if x not in first_category]

chart = (
    alt.Chart(cat_presence_plot)
    .mark_circle(size=130)
    .encode(
        x=alt.X(
            "census:O",
            title="Census",
            sort=["2001", "2011", "2021"],
            axis=alt.Axis(labelAngle=0)
        ),
        y=alt.Y(
            "class_name_wrapped:N",
            title=None,
            sort=y_order,
            axis=alt.Axis(
                labelLimit=500,
                labelFontSize=10
            )
        ),
        tooltip=[
            alt.Tooltip("class_name:N", title="Category"),
            alt.Tooltip("census:O", title="Census")
        ]
    )
    .properties(
        width=400,
        height=700,
        title="Category availability by Census"
    )
)

chart

alt.Chart(...)

:::

# Visualisation

## Household composition

### Bradford vs England

In [5]:
def prepare_household_views(
    df,
    area_names=None,
    category_col="class_name",
    main_cat_col="main_category",
    sub_cat_col="sub_category",
    value_col="pct",
    total_row="Total: All households",
):
    df_plot = df.copy()
    df_plot["census"] = df_plot["census"].astype(str)

    if area_names is not None:
        df_plot = df_plot.loc[
            df_plot["geography_name"].isin(area_names)
        ].copy()

    views = []

    sub1_col = f"{sub_cat_col}1"
    sub2_col = f"{sub_cat_col}2"

    # -------------------------
    # Main categories
    # -------------------------
    main = df_plot.loc[
        (df_plot[category_col] != total_row) &
        (df_plot[category_col] == df_plot[main_cat_col])
    ].copy()

    main["view"] = "Main categories"
    main["display_category"] = main[main_cat_col]
    main["display_level"] = main_cat_col

    views.append(main)

    # -------------------------
    # Auto-generated breakdown views
    # -------------------------
    main_categories_with_breakdown = (
        df_plot
        .loc[df_plot[sub1_col].notna(), main_cat_col]
        .dropna()
        .drop_duplicates()
        .tolist()
    )

    for main_cat in main_categories_with_breakdown:
        temp_all = df_plot.loc[
            (df_plot[main_cat_col] == main_cat) &
            (df_plot[sub1_col].notna())
        ].copy()

        if temp_all.empty:
            continue

        # Check whether this main category has deeper breakdowns
        has_sub2 = (
            sub2_col in temp_all.columns
            and temp_all[sub2_col].notna().any()
        )

        if has_sub2:
            # Use immediate sub_category1 rows only
            # e.g. Single family household: Married couple
            # but not Single family household: Married couple: No children
            temp = temp_all.loc[
                temp_all[sub2_col].isna()
            ].copy()
        else:
            # If there is no sub_category2, then sub_category1 is already the finest available breakdown
            temp = temp_all.copy()

        # Avoid creating a breakdown view if there is only one category
        n_display_categories = temp[sub1_col].dropna().nunique()

        if n_display_categories <= 1:
            continue

        temp["view"] = f"Breakdown: {main_cat}"
        temp["display_category"] = temp[sub1_col]
        temp["display_level"] = sub1_col

        views.append(temp)

    out = pd.concat(views, ignore_index=True)

    out = out.loc[
        out[value_col].notna()
    ].copy()

    return out

In [6]:
def plot_household_area_compare(
    df,
    area_names=("Bradford", "England"),
    category_col="class_name",
    main_cat_col="main_category",
    sub_cat_col="sub_category",
    value_col="pct",
    title="Household categories comparison",
):
    area_names = list(area_names)

    df_plot = prepare_household_views(
        df=df,
        area_names=area_names,
        category_col=category_col,
        main_cat_col=main_cat_col,
        sub_cat_col=sub_cat_col,
        value_col=value_col,
    )

    view_options = (
        df_plot["view"]
        .drop_duplicates()
        .tolist()
    )

    # Make sure Main categories is first
    view_options = (
        ["Main categories"]
        + [v for v in view_options if v != "Main categories"]
    )

    view_dropdown = alt.binding_select(
        options=view_options,
        name="View: "
    )

    view_select = alt.selection_point(
        fields=["view"],
        bind=view_dropdown,
        value="Main categories"
    )

    chart = (
        alt.Chart(df_plot)
        .mark_bar()
        .encode(
            x=alt.X(
                "geography_name:N",
                title=None,
                sort=area_names,
                axis=alt.Axis(labelAngle=0),
            ),
            y=alt.Y(
                f"{value_col}:Q",
                title="Percentage of households (%)",
            ),
            color=alt.Color(
                "display_category:N",
                title="Category",
            ),
            column=alt.Column(
                "census:O",
                title="Census",
                sort=["2001", "2011", "2021"],
            ),
            tooltip=[
                alt.Tooltip("geography_name:N", title="Area"),
                alt.Tooltip("census:O", title="Census"),
                alt.Tooltip("view:N", title="View"),
                alt.Tooltip("display_category:N", title="Displayed category"),
                alt.Tooltip(category_col + ":N", title="Full category"),
                alt.Tooltip("number:Q", title="Number", format=",.0f"),
                alt.Tooltip(value_col + ":Q", title="Percentage", format=".1f"),
            ],
        )
        .add_params(view_select)
        .transform_filter(view_select)
        .properties(
            width=300,
            height=420,
            title=title,
        )
    )

    return chart

In [7]:
plot_household_area_compare(
    df_composition,
    area_names=("Bradford", "England"),
    title="Household composition: Bradford vs England"
)

alt.Chart(...)

### Within Bradford

In [8]:
def prepare_bradford_ward_views(
    df,
    geography_type="2025 ward",
    category_col="class_name",
    main_cat_col="main_category",
    sub_cat_col="sub_category",
    value_col="pct",
    total_row="Total: All households",
    top_n=10,
):
    # keep ward-level data only
    df_ward = df.loc[
        df["geography_type"] == geography_type
    ].copy()

    df_ward["census"] = df_ward["census"].astype(str)

    # create category views using existing hierarchy logic
    df_views = prepare_household_views(
        df=df_ward,
        area_names=None,
        category_col=category_col,
        main_cat_col=main_cat_col,
        sub_cat_col=sub_cat_col,
        value_col=value_col,
        total_row=total_row,
    )

    # calculate ward-level change between first and latest census
    change_base = (
        df_views
        .loc[df_views["view"] == "Main categories"]
        .pivot_table(
            index=["geography_name", "display_category"],
            columns="census",
            values=value_col,
            aggfunc="sum"
        )
        .reset_index()
    )

    # only calculate change where both years exist
    if {"2001", "2021"}.issubset(change_base.columns):
        change_base["abs_change_2001_2021"] = (
            change_base["2021"] - change_base["2001"]
        ).abs()
    else:
        census_cols = [
            c for c in change_base.columns
            if c not in ["geography_name", "display_category"]
        ]
        census_cols = sorted(census_cols)

        change_base["abs_change_2001_2021"] = (
            change_base[census_cols[-1]] - change_base[census_cols[0]]
        ).abs()

    ward_change = (
        change_base
        .groupby("geography_name", as_index=False)["abs_change_2001_2021"]
        .max()
        .sort_values("abs_change_2001_2021", ascending=False)
    )

    top_wards = ward_change.head(top_n)["geography_name"].tolist()

    # create ward group
    df_views["ward_group"] = df_views["geography_name"].where(
        df_views["geography_name"].isin(top_wards),
        "Other wards"
    )

    # for dropdown:
    # 1. Top 10 changed wards
    # 2. each individual ward
    top_view = df_views.loc[
        df_views["geography_name"].isin(top_wards)
    ].copy()

    top_view["ward_selection"] = f"Top {top_n} changed wards"

    individual_views = []

    for ward in sorted(df_views["geography_name"].dropna().unique()):
        temp = df_views.loc[
            df_views["geography_name"] == ward
        ].copy()

        temp["ward_selection"] = ward
        individual_views.append(temp)

    out = pd.concat([top_view] + individual_views, ignore_index=True)

    return out

In [9]:
def plot_bradford_wards(
    df,
    geography_type="2025 ward",
    category_col="class_name",
    main_cat_col="main_category",
    sub_cat_col="sub_category",
    value_col="pct",
    top_n=10,
    title="Household composition within Bradford wards",
):
    df_plot = prepare_bradford_ward_views(
        df=df,
        geography_type=geography_type,
        category_col=category_col,
        main_cat_col=main_cat_col,
        sub_cat_col=sub_cat_col,
        value_col=value_col,
        top_n=top_n,
    )

    # dropdown 1: category view
    view_options = df_plot["view"].drop_duplicates().tolist()
    view_options = (
        ["Main categories"]
        + sorted([v for v in view_options if v != "Main categories"])
    )

    view_dropdown = alt.binding_select(
        options=view_options,
        name="View: "
    )

    view_select = alt.selection_point(
        fields=["view"],
        bind=view_dropdown,
        value="Main categories"
    )

    # dropdown 2: ward selection
    ward_options = df_plot["ward_selection"].drop_duplicates().tolist()

    first_option = f"Top {top_n} changed wards"
    ward_options = (
        [first_option]
        + sorted([w for w in ward_options if w != first_option])
    )

    ward_dropdown = alt.binding_select(
        options=ward_options,
        name="Ward selection: "
    )

    ward_select = alt.selection_point(
        fields=["ward_selection"],
        bind=ward_dropdown,
        value=first_option
    )

    chart = (
        alt.Chart(df_plot)
        .mark_bar()
        .encode(
            x=alt.X(
                "census:O",
                title="Census",
                sort=["2001", "2011", "2021"],
                axis=alt.Axis(labelAngle=0),
            ),
            y=alt.Y(
                f"{value_col}:Q",
                title="Percentage of households (%)",
            ),
            color=alt.Color(
                "display_category:N",
                title="Category",
            ),
            column=alt.Column(
                "geography_name:N",
                title=None,
                sort=alt.SortField(
                    field=value_col,
                    order="descending"
                ),
            ),
            tooltip=[
                alt.Tooltip("geography_name:N", title="Ward"),
                alt.Tooltip("census:O", title="Census"),
                alt.Tooltip("view:N", title="View"),
                alt.Tooltip("display_category:N", title="Displayed category"),
                alt.Tooltip(category_col + ":N", title="Full category"),
                alt.Tooltip("number:Q", title="Number", format=",.0f"),
                alt.Tooltip(value_col + ":Q", title="Percentage", format=".1f"),
            ],
        )
        .add_params(view_select, ward_select)
        .transform_filter(view_select)
        .transform_filter(ward_select)
        .properties(
            width=130,
            height=320,
            title=title,
        )
    )

    return chart

In [10]:
plot_bradford_wards(
    df_composition,
    top_n=10,
    title="Household composition within Bradford wards"
)

alt.Chart(...)

## Household size

### Bradford vs England

In [11]:
def plot_bradford_vs_england(
    df,
    category_col="class_name",
    value_col="pct",
    area_name="Bradford",
    england_name="England",
    title="Bradford vs England",
    exclude_total=True
):
    df_plot = df.copy()

    if exclude_total:
        df_plot = df_plot.loc[
            df_plot[category_col] != "Total: All households"
        ].copy()

    df_plot["census"] = df_plot["census"].astype(str)

    df_plot = df_plot.loc[
        df_plot["geography_name"].isin([area_name, england_name])
    ].copy()

    chart = (
        alt.Chart(df_plot)
        .mark_bar()
        .encode(
            x=alt.X(
                "geography_name:N",
                title=None,
                sort=[area_name, england_name],
                axis=alt.Axis(labelAngle=0),
            ),
            y=alt.Y(
                f"{value_col}:Q",
                title="Percentage of households (%)"
            ),
            color=alt.Color(
                f"{category_col}:N",
                title="Category"
            ),
            column=alt.Column(
                "census:O",
                title="Census"
            ),
            tooltip=[
                alt.Tooltip("geography_name:N", title="Area"),
                alt.Tooltip("census:O", title="Census"),
                alt.Tooltip(f"{category_col}:N", title="Category"),
                alt.Tooltip("number:Q", title="Number", format=",.0f"),
                alt.Tooltip(f"{value_col}:Q", title="Percentage", format=".1f"),
            ]
        )
        .properties(
            width=300,
            height=800,
            title=title
        )
    )

    return chart

In [12]:
plot_bradford_vs_england(
    df_size,
    title="Household size: Bradford vs England"
)

alt.Chart(...)

### Within Bradford

In [13]:
def prepare_household_size_ward_data(
    df,
    geography_type="2025 ward",
    category_col="class_name",
    value_col="pct",
    total_row="Total: All households",
    top_n=10,
):
    df_ward = df.loc[
        df["geography_type"] == geography_type
    ].copy()

    df_ward["census"] = df_ward["census"].astype(str)

    df_ward = df_ward.loc[
        df_ward[category_col] != total_row
    ].copy()

    # calculate change between 2001 and 2021
    change_base = (
        df_ward
        .pivot_table(
            index=["geography_name", category_col],
            columns="census",
            values=value_col,
            aggfunc="sum"
        )
        .reset_index()
    )

    if {"2001", "2021"}.issubset(change_base.columns):
        change_base["abs_change"] = (
            change_base["2021"] - change_base["2001"]
        ).abs()
    else:
        census_cols = [
            c for c in change_base.columns
            if c not in ["geography_name", category_col]
        ]
        census_cols = sorted(census_cols)

        change_base["abs_change"] = (
            change_base[census_cols[-1]] - change_base[census_cols[0]]
        ).abs()

    ward_change = (
        change_base
        .groupby("geography_name", as_index=False)["abs_change"]
        .max()
        .sort_values("abs_change", ascending=False)
    )

    top_wards = ward_change.head(top_n)["geography_name"].tolist()

    # top N changed wards view
    top_view = df_ward.loc[
        df_ward["geography_name"].isin(top_wards)
    ].copy()

    top_view["ward_selection"] = f"Top {top_n} changed wards"

    # individual ward views
    individual_views = []

    for ward in sorted(df_ward["geography_name"].dropna().unique()):
        temp = df_ward.loc[
            df_ward["geography_name"] == ward
        ].copy()

        temp["ward_selection"] = ward
        individual_views.append(temp)

    out = pd.concat([top_view] + individual_views, ignore_index=True)

    return out

In [14]:
def plot_household_size_wards(
    df,
    geography_type="2025 ward",
    category_col="class_name",
    value_col="pct",
    top_n=10,
    title="Household size within Bradford wards",
):
    df_plot = prepare_household_size_ward_data(
        df=df,
        geography_type=geography_type,
        category_col=category_col,
        value_col=value_col,
        top_n=top_n,
    )

    first_option = f"Top {top_n} changed wards"

    ward_options = df_plot["ward_selection"].drop_duplicates().tolist()
    ward_options = (
        [first_option]
        + sorted([w for w in ward_options if w != first_option])
    )

    ward_select = alt.selection_point(
        fields=["ward_selection"],
        bind=alt.binding_select(
            options=ward_options,
            name="Ward selection: "
        ),
        value=first_option
    )

    chart = (
        alt.Chart(df_plot)
        .mark_bar()
        .encode(
            x=alt.X(
                f"{value_col}:Q",
                title="Percentage of households (%)",
            ),
            y=alt.Y(
                "geography_name:N",
                title=None,
                sort="-x",
            ),
            color=alt.Color(
                f"{category_col}:N",
                title="Household size",
            ),
            row=alt.Row(
                "census:O",
                title="Census",
                sort=["2001", "2011", "2021"],
            ),
            tooltip=[
                alt.Tooltip("geography_name:N", title="Ward"),
                alt.Tooltip("census:O", title="Census"),
                alt.Tooltip(f"{category_col}:N", title="Household size"),
                alt.Tooltip("number:Q", title="Number", format=",.0f"),
                alt.Tooltip(f"{value_col}:Q", title="Percentage", format=".1f"),
            ],
        )
        .add_params(ward_select)
        .transform_filter(ward_select)
        .properties(
            width=620,
            height=280,
            title=title,
        )
    )

    return chart

In [15]:
plot_household_size_wards(
    df_size,
    top_n=10,
    title="Household size within Bradford wards"
)

alt.Chart(...)